# AIDA 2156 Final Project – 02 Processing Times and Process Steps

**Data version 2** (`shared/synthetic-data/`), cleaning version 1.1.

**Question:** How long do onboarding requests take, in working time, from submission until the responsible team finishes the setup? How is that time spread overall and by task type, and which stage of the workflow takes the longest?

This is steps 2 and 3 of the final project, which asks where delays and incomplete handoffs appear in the workflow and how reliable those findings are. Notebook 01 described the data's quality and remaining limitations; this notebook measures time.

This notebook is **descriptive**. It summarises the generated records and doesn't test whether differences are larger than chance; confidence intervals that keep each employee's requests together come in notebook 03. Several requests can belong to the same employee and share circumstances, so the requests are not independent.

All records are synthetic. Times describe a simulated workflow, not real HR performance or staff effort.

In [1]:
import hashlib
import importlib.util
import json
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
OUTPUT_DIR = Path("../outputs")
FIGURE_DIR = Path("../figures")
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

read = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False)
requests = read(SD / "cleaning/output/requests_cleaned.csv")        # official cleaned requests
cleaning_info = json.loads((SD / "cleaning/output/requests_cleaned_info.json").read_text())

sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sha(SD / "cleaning/output/requests_cleaned.csv") == cleaning_info["output_sha256"], \
    "The cleaned file doesn't match its cleaning record"

# The working calendar comes from the generator itself, so this notebook and the data use the same rules
spec = importlib.util.spec_from_file_location("generator", SD / "requests/generate_requests.py")
generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator)
EXTRACT_AT = pd.Timestamp(generator.EXTRACT_AT)
CLOSURES = set(generator.HOLIDAYS)
DAY_START, DAY_END = generator.DAY_START, generator.DAY_END
DAY_HOURS = (datetime.combine(datetime.min.date(), DAY_END)
             - datetime.combine(datetime.min.date(), DAY_START)).total_seconds() / 3600
assert DAY_HOURS > 0, "Working-day length must be positive"

# Record the calendar actually used, with the generator's fingerprint. No earlier record holds a generator
# fingerprint to compare against, so this file is the reference for later notebooks.
calendar_record = {
    "generator_file": "shared/synthetic-data/requests/generate_requests.py",
    "generator_sha256": sha(SD / "requests/generate_requests.py"),
    "extract_at": f"{EXTRACT_AT:%Y-%m-%d %H:%M}",
    "day_start": f"{DAY_START:%H:%M}",
    "day_end": f"{DAY_END:%H:%M}",
    "day_hours": DAY_HOURS,
    "closure_dates": [str(d) for d in sorted(CLOSURES)],
}
(OUTPUT_DIR / "working_calendar.json").write_text(json.dumps(calendar_record, indent=2))

TIME_COLUMNS = ["submitted_at", "info_complete_at", "assigned_at", "team_completed_at", "completion_confirmed_at"]
for c in TIME_COLUMNS:
    requests[c] = pd.to_datetime(requests[c].replace("", None), format="%Y-%m-%dT%H:%M")

print(f"Data version {DATA_VERSION}; cleaning version {cleaning_info['cleaning_version']}; extract {EXTRACT_AT:%Y-%m-%d %H:%M}")
print(f"{len(requests):,} cleaned requests; working day {DAY_START:%H:%M}-{DAY_END:%H:%M} ({DAY_HOURS} hours); "
      f"{len(CLOSURES)} simulated closure dates")
print(f"Generator fingerprint {calendar_record['generator_sha256'][:12]}… saved with the calendar to working_calendar.json")

Data version 2; cleaning version 1.1; extract 2026-09-29 17:00
4,364 cleaned requests; working day 08:00-16:30 (8.5 hours); 24 simulated closure dates
Generator fingerprint 118e70dc53ff… saved with the calendar to working_calendar.json


## A. Definitions

These are fixed before any duration is summarised.

**Working calendar.** Monday to Friday, 08:00–16:30, excluding the simulated Pinecrest closure dates (taken from the generator). Coordinators and teams only act within these hours.

**Working time.** The time between two moments that falls inside the working calendar, in hours. It is reported in **working-day equivalents**: working hours ÷ the length of the working day (8.5 hours). Because working time only counts hours inside the calendar, a request submitted after hours effectively starts counting at the next working moment (for example, 08:00 on Monday for a request sent on Friday evening); counting from the actual submission time gives exactly the same working duration.

**Elapsed calendar time.** The full time from the actual submission timestamp, including nights, weekends and closures, in calendar days. This is the requester's total wait.

**Processing time.** From submission (`submitted_at`, cleaned) to the moment the responsible team finished the setup (`team_completed_at`). Team completion is the moment that matters for the start-date deadline. The coordinator's later confirmation is reported separately as the **confirmation lag**.

**Stages.** Processing time splits into three stages that add up to the total:

1. **Information check:** submission → information complete. This includes waiting for the requester when information was missing or conflicting.
2. **Assignment wait:** information complete → assigned to the team.
3. **Team processing:** assigned → team finished.

**Which requests are summarised.** Completed-request summaries include requests with submission, information-complete, assignment and team-completion timestamps, with team completion no later than the extract date. Coordinator confirmation is not required, so requests still awaiting confirmation are included. The confirmation lag uses a separate subset with both team-completion and confirmation timestamps. Requests still unfinished at the extract date are **right-censored**: their processing time is at least the time so far, but not known. They are excluded from the completed-request summaries, counted, and shown separately with their time so far as a lower bound. Completed-request summaries therefore describe requests finished by the extract date, and can under-represent long-running requests; the size of that effect is not known. Completed requests whose submission time is still unresolved after cleaning would be a separate exclusion; they are counted below.

**Unusually long.** A request is called unusually long in this dataset if its processing time is above the upper Tukey fence, Q3 + 1.5 × IQR. One overall fence is calculated from all completed requests and applied unchanged when task types are compared, so each type's count answers the same question (how many of its requests are long by the dataset-wide standard). Separate fences per task type would answer a different question. This is an exploratory description of the distribution, not an operational failure.

In [2]:
def is_working_day(d):
    return d.weekday() < 5 and d not in CLOSURES


def working_hours(start, end):
    # Hours between two timestamps that fall inside the working calendar
    if pd.isna(start) or pd.isna(end):
        return np.nan
    if end < start:
        raise ValueError(f"End precedes start: {start} → {end}")
    if end == start:
        return 0.0
    total, day = 0.0, start.date()
    while day <= end.date():
        if is_working_day(day):
            open_, close = datetime.combine(day, DAY_START), datetime.combine(day, DAY_END)
            overlap = (min(end, close) - max(start, open_)).total_seconds()
            total += max(overlap, 0) / 3600
        day += timedelta(days=1)
    return total


# Hand-made tests of the working-time calculation
T = pd.Timestamp
assert T("2026-04-03").date() in CLOSURES, "The closure test assumes Fri 3 Apr 2026 is a simulated closure"
assert is_working_day(T("2026-04-06").date()), "The closure test assumes Mon 6 Apr 2026 is a working day"
tests = pd.DataFrame([
    ("same working day", T("2026-03-03 09:00"), T("2026-03-03 11:30"), 2.5),
    ("overnight", T("2026-03-03 15:30"), T("2026-03-04 09:00"), 2.0),
    ("over a weekend", T("2026-03-06 16:00"), T("2026-03-09 08:30"), 1.0),
    ("after-hours submission", T("2026-03-06 21:00"), T("2026-03-09 10:00"), 2.0),
    ("after-hours = next working moment", T("2026-03-09 08:00"), T("2026-03-09 10:00"), 2.0),
    ("over a simulated closure date (Fri 3 Apr 2026)", T("2026-04-02 16:00"), T("2026-04-06 09:00"), 1.5),
    ("whole working day", T("2026-03-03 08:00"), T("2026-03-03 16:30"), DAY_HOURS),
    ("start equals end", T("2026-03-03 10:00"), T("2026-03-03 10:00"), 0.0),
], columns=["case", "start", "end", "expected_hours"])
tests["calculated_hours"] = [working_hours(s, e) for s, e in zip(tests["start"], tests["end"])]
tests["passed"] = np.isclose(tests["calculated_hours"], tests["expected_hours"])
tests.to_csv(OUTPUT_DIR / "working_time_tests.csv", index=False)
display(tests)
assert tests["passed"].all(), "The working-time calculation failed a hand-made test"

# Reversed timestamps must be rejected, not turned into zero
try:
    working_hours(T("2026-03-03 10:00"), T("2026-03-03 09:00"))
except ValueError:
    print("Reversed timestamps are rejected, as intended.")
else:
    raise AssertionError("Reversed timestamps should be rejected")

,case,start,end,expected_hours,calculated_hours,passed
0,same working day,2026-03-03 09:00:00,2026-03-03 11:30:00,2.5,2.5,True
1,overnight,2026-03-03 15:30:00,2026-03-04 09:00:00,2.0,2.0,True
2,over a weekend,2026-03-06 16:00:00,2026-03-09 08:30:00,1.0,1.0,True
3,after-hours submission,2026-03-06 21:00:00,2026-03-09 10:00:00,2.0,2.0,True
4,after-hours = next working moment,2026-03-09 08:00:00,2026-03-09 10:00:00,2.0,2.0,True
5,over a simulated closure date (Fri 3 Apr 2026),2026-04-02 16:00:00,2026-04-06 09:00:00,1.5,1.5,True
6,whole working day,2026-03-03 08:00:00,2026-03-03 16:30:00,8.5,8.5,True
7,start equals end,2026-03-03 10:00:00,2026-03-03 10:00:00,0.0,0.0,True


Reversed timestamps are rejected, as intended.
